# TASK 1: DATA MERGE FOR PRODUCT RECOMMENDATION
**Goal:** Merge customer_transactions and customer_social_profiles to predict product purchases

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
print("✓ Libraries loaded")

## 1. LOAD DATA

In [ ]:
transactions_df = pd.read_csv('data/raw/customer_transactions.csv')
social_df = pd.read_csv('data/raw/customer_social_profiles.csv')

print(f"Transactions: {transactions_df.shape}")
print(f"Social: {social_df.shape}")
display(transactions_df.head())
display(social_df.head())

## 2. EXPLORATORY DATA ANALYSIS

In [ ]:
print("=== SUMMARY STATISTICS ===")
print("\nTransactions:")
print(transactions_df.describe())
print("\nSocial:")
print(social_df.describe())
print("\n=== DATA TYPES ===")
print("\nTransactions:", transactions_df.dtypes.to_dict())
print("\nSocial:", social_df.dtypes.to_dict())

In [ ]:
print("=== DATA QUALITY ===")
print("\nTransactions - Missing:")
print(transactions_df.isnull().sum())
print(f"Duplicates: {transactions_df.duplicated().sum()}")
print("\nSocial - Missing:")
print(social_df.isnull().sum())
print(f"Duplicates: {social_df.duplicated().sum()}")

In [ ]:
# PLOT 1: Product Category Distribution
fig, ax = plt.subplots(figsize=(10, 5))
transactions_df['product_category'].value_counts().plot(kind='bar', ax=ax, color='skyblue')
ax.set_title('Product Category Distribution', fontsize=14)
ax.set_xlabel('Category')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# PLOT 2: Purchase Amount Distribution with Outliers
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(transactions_df['purchase_amount'], bins=30, color='coral', edgecolor='black')
axes[0].set_title('Purchase Amount Distribution')
axes[0].set_xlabel('Amount')
axes[0].set_ylabel('Frequency')
axes[1].boxplot(transactions_df['purchase_amount'])
axes[1].set_title('Purchase Amount - Outlier Detection')
axes[1].set_ylabel('Amount')
plt.tight_layout()
plt.show()

In [ ]:
# PLOT 3: Social Media Engagement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(social_df['engagement_score'], bins=30, color='lightgreen', edgecolor='black')
axes[0].set_title('Engagement Score Distribution')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Frequency')
social_df['social_media_platform'].value_counts().plot(kind='bar', ax=axes[1], color='plum')
axes[1].set_title('Platform Distribution')
axes[1].set_xlabel('Platform')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.show()

In [ ]:
# PLOT 4: Correlation Analysis
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
trans_corr = transactions_df[['purchase_amount', 'customer_rating']].corr()
sns.heatmap(trans_corr, annot=True, cmap='coolwarm', ax=axes[0], vmin=-1, vmax=1)
axes[0].set_title('Transactions Correlation')
social_corr = social_df[['engagement_score', 'purchase_interest_score']].corr()
sns.heatmap(social_corr, annot=True, cmap='coolwarm', ax=axes[1], vmin=-1, vmax=1)
axes[1].set_title('Social Correlation')
plt.tight_layout()
plt.show()

## 3. DATA CLEANING

In [ ]:
# Handle missing ratings
missing_before = transactions_df['customer_rating'].isnull().sum()
transactions_df['customer_rating'].fillna(transactions_df['customer_rating'].median(), inplace=True)
print(f"Missing ratings: {missing_before} → {transactions_df['customer_rating'].isnull().sum()}")

# Remove duplicates
transactions_df.drop_duplicates(inplace=True)
social_df.drop_duplicates(inplace=True)
print(f"✓ Cleaned - Transactions: {transactions_df.shape}, Social: {social_df.shape}")

## 4. ALIGN CUSTOMER IDs

In [ ]:
# Remove 'A' prefix from social IDs to match transaction IDs
social_df['customer_id'] = social_df['customer_id_new'].str.replace('A', '').astype(int)
transactions_df['customer_id'] = transactions_df['customer_id_legacy']

print(f"Unique customers - Transactions: {transactions_df['customer_id'].nunique()}")
print(f"Unique customers - Social: {social_df['customer_id'].nunique()}")

## 5. FEATURE ENGINEERING

In [ ]:
# Aggregate transaction features per customer
transaction_features = transactions_df.groupby('customer_id').agg({
    'purchase_amount': ['mean', 'sum', 'count'],
    'customer_rating': 'mean',
    'product_category': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
}).reset_index()

transaction_features.columns = ['customer_id', 'avg_purchase_amount', 'total_spent', 
                                 'transaction_count', 'avg_rating', 'most_purchased_category']

print("Transaction features:")
display(transaction_features.head())

In [ ]:
# Aggregate social features per customer
social_features = social_df.groupby('customer_id').agg({
    'engagement_score': 'mean',
    'purchase_interest_score': 'mean',
    'social_media_platform': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0],
    'review_sentiment': lambda x: x.mode()[0] if len(x.mode()) > 0 else x.iloc[0]
}).reset_index()

social_features.columns = ['customer_id', 'avg_engagement_score', 'avg_purchase_interest', 
                           'primary_platform', 'dominant_sentiment']

print("Social features:")
display(social_features.head())

## 6. MERGE DATASETS

In [ ]:
# Inner join on customer_id (keeps only customers in both datasets)
merged_df = transaction_features.merge(social_features, on='customer_id', how='inner')

print(f"✓ Merged: {merged_df.shape}")
display(merged_df.head())

## 7. MERGE VALIDATION

In [ ]:
print("=== MERGE VALIDATION ===")
print(f"Before - Transactions: {len(transaction_features)}, Social: {len(social_features)}")
print(f"After - Merged: {len(merged_df)}")
print(f"\nMissing values:\n{merged_df.isnull().sum()}")
print(f"\nDuplicates: {merged_df.duplicated().sum()}")
print("\n✓ Merge successful - all customers matched")

## 8. ADDITIONAL FEATURES

In [ ]:
# Create derived features
merged_df['engagement_x_interest'] = merged_df['avg_engagement_score'] * merged_df['avg_purchase_interest']
merged_df['spending_per_transaction'] = merged_df['total_spent'] / merged_df['transaction_count']
merged_df['sentiment_encoded'] = merged_df['dominant_sentiment'].map({'Positive': 1, 'Neutral': 0, 'Negative': -1})

# One-hot encode platform
platform_dummies = pd.get_dummies(merged_df['primary_platform'], prefix='platform')
merged_df = pd.concat([merged_df, platform_dummies], axis=1)

print(f"✓ Final shape: {merged_df.shape}")
print(f"Features: {list(merged_df.columns)}")

## 9. FINAL SUMMARY

In [ ]:
print("=== TASK 1 COMPLETE ===")
print(f"Customers: {len(merged_df)}")
print(f"Features: {merged_df.shape[1]}")
print(f"\nTarget (most_purchased_category):\n{merged_df['most_purchased_category'].value_counts()}")
print("\nNumerical summary:")
display(merged_df.describe())

In [ ]:
# Final visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(merged_df['avg_engagement_score'], merged_df['avg_purchase_interest'], 
                c=merged_df['sentiment_encoded'], cmap='RdYlGn', alpha=0.6)
axes[0].set_xlabel('Engagement Score')
axes[0].set_ylabel('Purchase Interest')
axes[0].set_title('Engagement vs Interest (by sentiment)')
merged_df.groupby('most_purchased_category')['total_spent'].mean().plot(kind='bar', ax=axes[1], color='teal')
axes[1].set_title('Avg Spending by Category')
axes[1].set_xlabel('Category')
axes[1].set_ylabel('Avg Spent')
plt.tight_layout()
plt.show()

## 10. SAVE DATASET

In [ ]:
merged_df.to_csv('data/processed/merged_dataset.csv', index=False)
print("✓ Saved: data/processed/merged_dataset.csv")
print(f"✓ Shape: {merged_df.shape}")
print("✓ Ready for product recommendation model!")